In [ ]:
# %% [markdown]
# # Exp2 (simple) — Fourier-SHAP vs Kernel SHAP by Age (NO-AGE)
# - English-only prints/labels
# - ROC curve with filled areas (AUC: light gray, diagonal: dark gray)
# - Bar plots for MeanAbsSHAP (linear + log scales), underscores removed from feature labels
# - Per-age compute cost table (time + peak memory)
# - Composite figures for bins 0–3 and 4–7

# %%
# ---------------- Imports & Global Plot Style ----------------
import os, sys, time, math, warnings, tracemalloc
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # keep non-interactive backend for file saving
import matplotlib.pyplot as plt

# Increase global font sizes
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 13,
})

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             balanced_accuracy_score, accuracy_score, r2_score, roc_curve)
from sklearn.linear_model import ElasticNet, ElasticNetCV, RidgeCV

# Optional baselines
XGB_OK = True
CB_OK = True
try:
    from xgboost import XGBClassifier
except Exception as e:
    XGB_OK = False; xgb_import_err = e
try:
    from catboost import CatBoostClassifier
except Exception as e:
    CB_OK = False; cb_import_err = e

# SHAP
try:
    import shap
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "--quiet"])
    import shap

warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(suppress=True, linewidth=160)

# %%
# ---------------- Config & Small Utilities ----------------
@dataclass
class Config:
    csv_candidates: Tuple[str, ...] = ("./stroke_coded_binned_gender012.csv", "/mnt/data/stroke_coded_binned_gender012.csv")
    outdir: str = f"./OutExp2_{int(time.time())}"
    random_state: int = 42

    # Splits
    test_size: float = 0.15
    val_size: float = 0.15  # of the remainder

    # NN
    nn_hidden: Tuple[int, ...] = (256, 128, 64)
    nn_lr: float = 1e-3
    nn_l2: float = 1e-4
    nn_batch_size: int = 256
    nn_max_iter: int = 500
    nn_early_stopping: bool = True
    nn_n_iter_no_change: int = 30

    # Fourier (spectralPLUS)
    max_nonconst_modes_per_feature: int = 7
    dmax: int = 3
    TOP_MODES_PER_FEATURE_FOR_PAIRS: int = 5
    K1_UNI: int = 300
    K2_PAIR: int = 4000
    K3_TRIP: int = 2000
    STAB_B: int = 30
    STAB_SUBSAMPLE_FRAC: float = 0.7
    STAB_L1: float = 0.85
    STAB_ALPHA: float = 0.01
    STAB_FREQ_THR: float = 0.60
    STAB_MIN_KEEP: int = 100
    ENET_GRID_L1: Tuple[float, ...] = (0.3, 0.5, 0.7, 0.85, 0.95)
    ENET_N_ALPHAS: int = 120
    ENET_CV: int = 7
    RIDGE_ALPHAS: Tuple[float, ...] = tuple(np.logspace(-4, 2, 15))
    SUFFIX: str = "sPLUS"

    # Kernel SHAP per bin
    KERNEL_BIN_SUB_N: int = 100
    KERNEL_NSAMPLES: int | str = 2048

    # Age
    AGE_BINS: Tuple[int, ...] = tuple(range(8))
    TOPK_BARS: int = 20

cfg = Config()
os.makedirs(cfg.outdir, exist_ok=True)

def now(): return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
def logit(p, eps=1e-6): p = np.clip(p, eps, 1-eps); return np.log(p) - np.log1p(-p)

def safe_save_csv(df: pd.DataFrame, path: str, **kw):
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, **kw)

def safe_save_fig(path: str, fmt: str):
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.savefig(path, format=fmt, bbox_inches="tight")

def detect_target_column(df: pd.DataFrame) -> str:
    for c in ["stroke", "Stroke", "target", "label", "y"]:
        if c in df.columns: return c
    for c in df.columns[::-1]:
        u = df[c].dropna().unique()
        if len(set(u)) <= 2 and set(u).issubset({0,1}): return c
    raise ValueError("Target column not found (e.g., 'stroke').")

def drop_identifier_columns(cols: List[str]) -> List[str]:
    ID = {"id","patient_id","row_id","index","subject_id","record_id","case_id"}
    return [c for c in cols if c.lower() not in ID]

def make_one_hot_encoder():
    try:    return OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except: return OneHotEncoder(sparse=False, handle_unknown="ignore")

def tune_threshold(y_true, p, grid=200):
    from sklearn.metrics import f1_score, accuracy_score
    ts = np.linspace(0.01, 0.99, grid)
    best_t, best_f1, best_acc = 0.5, -1.0, 0.0
    for t in ts:
        yhat = (p >= t).astype(int)
        f1 = f1_score(y_true, yhat, zero_division=0)
        acc = accuracy_score(y_true, yhat)
        if f1 > best_f1 or (f1==best_f1 and acc>best_acc): best_t, best_f1, best_acc = t, f1, acc
    return best_t, {"F1": best_f1, "Acc": best_acc}

def class_weights_sample_weight(y):
    n=len(y); pos=y.sum(); neg=n-pos
    w_pos=0.5/(pos/n+1e-12); w_neg=0.5/(neg/n+1e-12)
    return np.where(y==1,w_pos,w_neg)

def find_age_column(df: pd.DataFrame) -> str:
    cands = [c for c in df.columns if "age" in c.lower()]
    if not cands: raise ValueError("Age column not found.")
    for c in cands:
        s=df[c]
        if np.issubdtype(s.dtype, np.number) and s.nunique(dropna=True)<=20: return c
    return cands[0]

def display_name(feat: str) -> str:
    """Pretty display: replace '_' with space and Title Case."""
    return feat.replace("_", " ").title()

# %%
# ---------------- Discrete Orthonormal Basis & Design ----------------
class DiscreteOrthonormalBasis:
    def __init__(self, Xtr: pd.DataFrame, max_nonconst_modes=7):
        self.features=list(Xtr.columns); self.L_cap=max_nonconst_modes
        self.cat_values={}; self.p_marginal={}; self.psi_tables={}
        for f in self.features:
            vals=np.sort(Xtr[f].dropna().unique()).astype(int)
            self.cat_values[f]=vals
            counts=Xtr[f].value_counts().reindex(vals, fill_value=0).to_numpy(float)
            p=counts/counts.sum(); self.p_marginal[f]=p
            m=len(vals); L=int(min(self.L_cap, max(m-1,0)))
            C=np.zeros((m,1+L)); C[:,0]=1.0
            for j in range(1,1+L):
                e=np.zeros(m); e[j]=1.0; C[:,j]=e-p
            psi=self._wgs(C,p); psi[:,0]=1.0; self.psi_tables[f]=psi
    @staticmethod
    def _win(u,v,p): return float(np.sum(p*u*v))
    def _wgs(self,C,p):
        r=C.shape[1]; Q=np.zeros_like(C)
        for j in range(r):
            v=C[:,j].copy()
            for k in range(j): v-= self._win(Q[:,k],v,p)*Q[:,k]
            nrm=math.sqrt(max(self._win(v,v,p),1e-15)); Q[:,j]=v/nrm
        for j in range(1,r): Q[:,j]-= self._win(Q[:,j], np.ones_like(p), p)
        return Q
    def eval_modes(self, xcol: np.ndarray, feat: str, nonconst_only=True):
        vals=self.cat_values[feat]; psi=self.psi_tables[feat]
        idx=np.vectorize({v:i for i,v in enumerate(vals)}.get)(xcol.astype(int))
        return psi[idx,1:] if nonconst_only else psi[idx,:]

def screening_ampliado(basis, Xtr, y_logit, cfg):
    feats=list(Xtr.columns)
    cache={f: basis.eval_modes(Xtr[f].to_numpy(), f, True) for f in feats}
    uni=[]; top_per_feat={}
    for f in feats:
        M=cache[f]; cs=[]
        for j in range(M.shape[1]):
            c=np.corrcoef(M[:,j], y_logit)[0,1]; c=0.0 if not np.isfinite(c) else abs(c)
            uni.append(((f,j+1),c)); cs.append((j+1,c))
        cs.sort(key=lambda t:t[1], reverse=True)
        top_per_feat[f]=[jj for jj,_ in cs[:cfg.TOP_MODES_PER_FEATURE_FOR_PAIRS]]
    uni.sort(key=lambda t:t[1], reverse=True)
    top_uni=[(u[0],) for u in uni[:cfg.K1_UNI]]

    pairs=[]
    if cfg.dmax>=2 and cfg.K2_PAIR>0:
        for i in range(len(feats)):
            f1=feats[i]
            for j in range(i+1,len(feats)):
                f2=feats[j]
                for j1 in top_per_feat[f1]:
                    v1=cache[f1][:,j1-1]
                    for j2 in top_per_feat[f2]:
                        v2=cache[f2][:,j2-1]
                        w=v1*v2
                        c=np.corrcoef(w, y_logit)[0,1]
                        if np.isfinite(c): pairs.append((((f1,j1),(f2,j2)), abs(c)))
        pairs.sort(key=lambda t:t[1], reverse=True)
    top_pair=[p[0] for p in pairs[:cfg.K2_PAIR]]

    triples=[]
    if cfg.dmax>=3 and cfg.K3_TRIP>0 and len(top_pair)>0:
        top_pair_l=top_pair[:min(len(top_pair),1500)]
        for (f1,j1),(f2,j2) in top_pair_l:
            for f3 in feats:
                if f3 in (f1,f2): continue
                for j3 in top_per_feat[f3][:3]:
                    w=cache[f1][:,j1-1]*cache[f2][:,j2-1]*cache[f3][:,j3-1]
                    c=np.corrcoef(w, y_logit)[0,1]
                    if np.isfinite(c): triples.append((((f1,j1),(f2,j2),(f3,j3)), abs(c)))
        triples.sort(key=lambda t:t[1], reverse=True)
    top_trip=[t[0] for t in triples[:cfg.K3_TRIP]]

    atoms=top_uni+top_pair+top_trip
    atoms=list(dict.fromkeys(atoms)); atoms.sort(key=lambda a:(len(a),a))
    return atoms

def build_design_matrix(basis, X, atoms, include_const=False):
    n=len(X); A=np.empty((n, len(atoms)+(1 if include_const else 0))); col0=0
    if include_const: A[:,0]=1.0; col0=1
    cache={f: basis.eval_modes(X[f].to_numpy(), f, True) for f in X.columns}
    for k, atom in enumerate(atoms):
        v=np.ones(n)
        for (f,j) in atom: v*= cache[f][:,j-1]
        A[:,col0+k]=v
    return A

def stability_selection(A, y, B=30, frac=0.7, l1_ratio=0.85, alpha=0.01, seed=42):
    rng=np.random.default_rng(seed); n,p=A.shape; counts=np.zeros(p,int)
    for _ in range(B):
        idx=rng.choice(n, size=int(max(10, frac*n)), replace=True)
        en=ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=True, max_iter=5000, random_state=seed)
        en.fit(A[idx], y[idx]); counts += (np.abs(en.coef_)>1e-10).astype(int)
    return counts/B

def debias_ridge(A,y,fit_intercept=True, alphas=None):
    if alphas is None: alphas=np.logspace(-4,2,15)
    rc=RidgeCV(alphas=alphas, fit_intercept=fit_intercept, cv=None); rc.fit(A,y)
    return rc.intercept_, rc.coef_

def fourier_shap_from_coeffs(coeffs, atoms, basis, X):
    feats=list(X.columns); n=len(X); c=coeffs[1:]
    cache={f: basis.eval_modes(X[f].to_numpy(), f, True) for f in feats}
    S=np.zeros((n,len(feats)))
    for k, atom in enumerate(atoms):
        supp=[f for (f,_) in atom]; d=len(supp)
        v=np.ones(n)
        for (f,j) in atom: v*= cache[f][:,j-1]
        contrib=c[k]/max(d,1)*v
        for f in supp: S[:,feats.index(f)] += contrib
    return pd.DataFrame(S, index=X.index, columns=feats)

def fourier_shap_excluding_feature(coeffs, atoms, basis, X, exclude_feat):
    feats=[f for f in X.columns if f!=exclude_feat]; n=len(X); c=coeffs[1:]
    mask=[all(f!=exclude_feat for (f,_) in a) for a in atoms]
    atoms_f=[a for a,m in zip(atoms,mask) if m]
    c_f=[ci for ci,m in zip(c,mask) if m]; c_f=np.array(c_f,float)
    cache={f: basis.eval_modes(X[f].to_numpy(), f, True) for f in X.columns}
    S=np.zeros((n,len(feats))); fmap={f:i for i,f in enumerate(feats)}
    for k, atom in enumerate(atoms_f):
        supp=[f for (f,_) in atom]; d=len(supp)
        v=np.ones(n)
        for (f,j) in atom: v*= cache[f][:,j-1]
        contrib=c_f[k]/max(d,1)*v
        for f in supp:
            if f==exclude_feat: continue
            S[:,fmap[f]] += contrib
    return pd.DataFrame(S, index=X.index, columns=feats)

class NNWithEncoder:
    def __init__(self, enc, clf): self.enc=enc; self.clf=clf
    def predict_proba(self, X): return self.clf.predict_proba(self.enc.transform(X))

def _extract_class1_shap(shap_values, n_sub, n_features):
    if isinstance(shap_values, list): arr=np.asarray(shap_values[1])
    else: arr=np.asarray(shap_values)
    if arr.ndim==2:
        if arr.shape==(n_sub,n_features): return arr
        if arr.shape==(n_features,n_sub): return arr.T
        raise ValueError(f"Unexpected 2D SHAP shape: {arr.shape}")
    if arr.ndim==3:
        for ax,s in enumerate(arr.shape):
            if s==2:
                cut=np.take(arr, indices=1, axis=ax)
                if cut.shape==(n_sub,n_features): return cut
                if cut.shape==(n_features,n_sub): return cut.T
        raise ValueError(f"Unexpected 3D SHAP shape: {arr.shape}")
    raise ValueError(f"Unsupported SHAP ndim: {arr.ndim}")

# %%
# ---------------- Load Data, Split & Train Models ----------------
print(f"[{now()}] Start. Output -> {cfg.outdir}")

csv_path=next((p for p in cfg.csv_candidates if os.path.exists(p)), None)
if csv_path is None: raise FileNotFoundError("Could not find 'stroke_coded_binned_gender012.csv'.")
df=pd.read_csv(csv_path)
y_col=detect_target_column(df); age_col=find_age_column(df)

features=drop_identifier_columns([c for c in df.columns if c!=y_col])
X=df[features].copy(); y=df[y_col].astype(int).to_numpy()

# Split
X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=cfg.test_size, stratify=y, random_state=cfg.random_state)
rel_val=cfg.val_size/(1.0-cfg.test_size)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=rel_val, stratify=y_tmp, random_state=cfg.random_state)

# One-hot
enc=make_one_hot_encoder(); Xtr=enc.fit_transform(X_tr); Xva=enc.transform(X_va); Xte=enc.transform(X_te)

# NN
nn=MLPClassifier(hidden_layer_sizes=cfg.nn_hidden, activation="relu", solver="adam",
                 alpha=cfg.nn_l2, batch_size=cfg.nn_batch_size, learning_rate_init=cfg.nn_lr,
                 max_iter=cfg.nn_max_iter, early_stopping=cfg.nn_early_stopping,
                 n_iter_no_change=cfg.nn_n_iter_no_change, random_state=cfg.random_state, verbose=False)
nn.fit(Xtr, y_tr, sample_weight=class_weights_sample_weight(y_tr))
prob=lambda X_: nn.predict_proba(X_)[:,1]
p_va=prob(Xva); p_te=prob(Xte)
t_star, vb=tune_threshold(y_va, p_va); yhat_te=(p_te>=t_star).astype(int)

metrics=[{
    "Model":"NN",
    "AUC_val": roc_auc_score(y_va, p_va),
    "AUC_test": roc_auc_score(y_te, p_te),
    "AP_val": average_precision_score(y_va, p_va),
    "AP_test": average_precision_score(y_te, p_te),
    "F1_val@best": vb["F1"], "Acc_val@best": vb["Acc"], "Thresh_val": t_star,
    "F1_test": f1_score(y_te, yhat_te, zero_division=0),
    "Acc_test": accuracy_score(y_te, yhat_te),
    "BalAcc_test": balanced_accuracy_score(y_te, yhat_te),
    "R2_test": r2_score(y_te, p_te)
}]

# Optional XGB
if XGB_OK:
    spw=(len(y_tr)-y_tr.sum())/max(1,y_tr.sum())
    xgb=XGBClassifier(n_estimators=600,max_depth=6,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
                      reg_lambda=1.0,objective="binary:logistic",eval_metric="auc",
                      scale_pos_weight=spw,random_state=cfg.random_state,n_jobs=-1)
    xgb.fit(Xtr,y_tr)
    pva=xgb.predict_proba(Xva)[:,1]; pte=xgb.predict_proba(Xte)[:,1]
    t, vb2=tune_threshold(y_va,pva); yhat=(pte>=t).astype(int)
    metrics.append({
        "Model":"XGBoost","AUC_val":roc_auc_score(y_va,pva),"AUC_test":roc_auc_score(y_te,pte),
        "AP_val":average_precision_score(y_va,pva),"AP_test":average_precision_score(y_te,pte),
        "F1_val@best":vb2["F1"],"Acc_val@best":vb2["Acc"],"Thresh_val":t,
        "F1_test":f1_score(y_te,yhat,zero_division=0),"Acc_test":accuracy_score(y_te,yhat),
        "BalAcc_test":balanced_accuracy_score(y_te,yhat),"R2_test":r2_score(y_te,pte)
    })
else:
    print(f"[WARN] xgboost not available: {xgb_import_err}")

# Optional CatBoost
if CB_OK:
    cw=(len(y_tr)-y_tr.sum())/max(1,y_tr.sum())
    cb=CatBoostClassifier(iterations=800,depth=6,learning_rate=0.05,l2_leaf_reg=3.0,
                          loss_function="Logloss",eval_metric="AUC",verbose=False,
                          random_seed=cfg.random_state,class_weights=[1.0,float(cw)])
    cb.fit(Xtr,y_tr)
    pva=cb.predict_proba(Xva)[:,1]; pte=cb.predict_proba(Xte)[:,1]
    t, vb3=tune_threshold(y_va,pva); yhat=(pte>=t).astype(int)
    metrics.append({
        "Model":"CatBoost","AUC_val":roc_auc_score(y_va,pva),"AUC_test":roc_auc_score(y_te,pte),
        "AP_val":average_precision_score(y_va,pva),"AP_test":average_precision_score(y_te,pte),
        "F1_val@best":vb3["F1"],"Acc_val@best":vb3["Acc"],"Thresh_val":t,
        "F1_test":f1_score(y_te,yhat,zero_division=0),"Acc_test":accuracy_score(y_te,yhat),
        "BalAcc_test":balanced_accuracy_score(y_te,yhat),"R2_test":r2_score(y_te,pte)
    })
else:
    print(f"[WARN] catboost not available: {cb_import_err}")

safe_save_csv(pd.DataFrame(metrics), os.path.join(cfg.outdir, "model_metrics.csv"), index=False)

In [10]:
# %%
# ---------------- ROC Plot (separate cell for plotting) ----------------
fpr, tpr, _ = roc_curve(y_te, p_te)
auc_val = roc_auc_score(y_te, p_te)
xs = np.linspace(0, 1, 200)

plt.figure(figsize=(7.8, 6.8))
# Fill diagonal area (dark gray)
plt.fill_between(xs, 0, xs, color="dimgray", alpha=0.35, label="Diagonal area")
# Fill ROC AUC area (light gray)
plt.fill_between(fpr, 0, tpr, color="lightgray", alpha=0.7, label="AUC area")
# Lines
#plt.plot(fpr, tpr, lw=2.5, label=f"ROC (AUC = {auc_val:.4f})")
plt.plot(fpr, tpr, lw=2.5, color="orange", label=f"ROC (AUC = {auc_val:.4f})")
plt.plot([0,1],[0,1], linestyle="--", lw=1.8, color="black", label="Chance")
plt.xlim([0,1]); plt.ylim([0,1])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on Test Set")
plt.legend(loc="lower right")
plt.tight_layout()
safe_save_fig(os.path.join(cfg.outdir, "ROC_Test.pdf"), fmt="pdf")
safe_save_fig(os.path.join(cfg.outdir, "ROC_Test.png"), fmt="png")
plt.close()

In [4]:
# %%
# ---------------- Fit Fourier surrogate on NN logits ----------------
basis=DiscreteOrthonormalBasis(X_tr, max_nonconst_modes=cfg.max_nonconst_modes_per_feature)
ytr_log=logit(prob(Xtr)); yva_log=logit(prob(Xva)); yte_log=logit(prob(Xte))

atoms=screening_ampliado(basis, X_tr, ytr_log, cfg)
def mat(Xd): return build_design_matrix(basis, Xd, atoms, include_const=False)
A_tr, A_va, A_te = mat(X_tr), mat(X_va), mat(X_te)

freq=stability_selection(A_tr, ytr_log, B=cfg.STAB_B, frac=cfg.STAB_SUBSAMPLE_FRAC,
                         l1_ratio=cfg.STAB_L1, alpha=cfg.STAB_ALPHA, seed=cfg.random_state)
thr=cfg.STAB_FREQ_THR; sel=np.where(freq>=thr)[0]
while len(sel)<cfg.STAB_MIN_KEEP and thr>0.3:
    thr-=0.05; sel=np.where(freq>=thr)[0]
atoms_sel=[atoms[i] for i in sel]
A_trs, A_vas, A_tes = A_tr[:,sel], A_va[:,sel], A_te[:,sel]

enet=ElasticNetCV(l1_ratio=list(cfg.ENET_GRID_L1), n_alphas=cfg.ENET_N_ALPHAS,
                  fit_intercept=True, cv=cfg.ENET_CV, max_iter=8000, random_state=cfg.random_state)
enet.fit(A_trs, ytr_log)
keep=(np.abs(enet.coef_)>1e-12)
if keep.sum()==0: keep=np.ones_like(keep, bool)
atoms_fin=[atoms_sel[i] for i,k in enumerate(keep) if k]
A_trk, A_vak, A_tek = A_trs[:,keep], A_vas[:,keep], A_tes[:,keep]

b0, beta = debias_ridge(A_trk, ytr_log, fit_intercept=True, alphas=cfg.RIDGE_ALPHAS)
pred_tr=b0 + A_trk@beta; pred_va=b0 + A_vak@beta; pred_te=b0 + A_tek@beta
diags=pd.DataFrame([{
    "R2_train": r2_score(ytr_log,pred_tr),
    "R2_val":   r2_score(yva_log,pred_va),
    "R2_test":  r2_score(yte_log,pred_te),
    "atoms_cand": len(atoms),
    "atoms_stable": len(sel),
    "atoms_final": int(beta.size)
}])
safe_save_csv(diags, os.path.join(cfg.outdir, "spectral_fit.csv"), index=False)

coeffs=np.concatenate(([b0], beta))
FS_full  = fourier_shap_from_coeffs(coeffs, atoms_fin, basis, X_te)
FS_noage = fourier_shap_excluding_feature(coeffs, atoms_fin, basis, X_te, exclude_feat=age_col)
safe_save_csv(FS_full,  os.path.join(cfg.outdir, "fs_full.csv"),  index=True)
safe_save_csv(FS_noage, os.path.join(cfg.outdir, "fs_noage.csv"), index=True)

C:\Users\roberto.morales\AppData\Local\anaconda3\envs\stroke-mlp2\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1622: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


In [5]:
# %%
# ---------------- Kernel SHAP per age bin (NO-AGE) — compute only ----------------
wrapped=NNWithEncoder(enc, nn)
try:
    masker=shap.maskers.Independent(X_tr)
    expl=shap.Explainer(wrapped.predict_proba, masker, algorithm="kernel", link="logit")
    sfx_short="IND"
except Exception:
    expl=shap.KernelExplainer(wrapped.predict_proba, X_tr, link="logit")
    sfx_short="TBG"

age_dir=os.path.join(cfg.outdir, f"age_NOAGE_{sfx_short}")
os.makedirs(os.path.abspath(age_dir), exist_ok=True)

age_map=df[age_col].to_dict()
idx_te=list(X_te.index)

FS_noage.index=FS_noage.index.astype(int)
rng=np.random.default_rng(cfg.random_state)
sizes=[]
costs=[]

Using 3355 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


In [6]:
# Per-bin precomputed artifacts to re-use in plotting cell
perbin_data = {}  # b -> dict with ff, kk, feats_plot, KS_eq size, N_F, etc.

for b in list(cfg.AGE_BINS):
    idx_bin=[i for i in idx_te if age_map.get(i, np.nan)==b]
    if len(idx_bin)==0:
        safe_save_csv(pd.DataFrame(), os.path.join(age_dir, f"bin{b}_rank.csv"), index=False)
        sizes.append({"bin":b, "N_test":0, "N_F":0, "N_K_raw":0, "N_K_eq":0})
        costs.append({"bin": b,
                      "Kernel_time_sec": np.nan, "Kernel_peak_MB": np.nan,
                      "Fourier_time_sec": np.nan, "Fourier_peak_MB": np.nan})
        perbin_data[b] = None
        continue

    # Fourier NO-AGE on bin (cost)
    tracemalloc.start()
    t0 = time.perf_counter()
    FSb=FS_noage.loc[[i for i in FS_noage.index if i in idx_bin]].copy()
    N_F=len(FSb)
    ff=FSb.abs().mean(axis=0)  # MeanAbsSHAP (Fourier)
    fourier_time = time.perf_counter() - t0
    _, fourier_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    fourier_peak_mb = fourier_peak / (1024*1024)

    # Kernel SHAP subset
    X_bin=X_te.loc[idx_bin]
    n_sub=min(cfg.KERNEL_BIN_SUB_N, len(X_bin))
    X_sub=X_bin.sample(n=n_sub, random_state=cfg.random_state)

    tracemalloc.start()
    t0 = time.perf_counter()
    shap_vals=expl.shap_values(X_sub, nsamples=cfg.KERNEL_NSAMPLES)
    sv1=_extract_class1_shap(shap_vals, n_sub=n_sub, n_features=X_sub.shape[1])
    KS_raw=pd.DataFrame(sv1, index=X_sub.index, columns=list(X_sub.columns))
    KSb=KS_raw.drop(columns=[age_col], errors="ignore").copy()
    kernel_time = time.perf_counter() - t0
    _, kernel_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    kernel_peak_mb = kernel_peak / (1024*1024)

    # Equalize N
    if N_F==0:
        KS_eq=KSb.iloc[0:0]
    else:
        if len(KSb)<N_F: take=rng.choice(KSb.index, size=N_F, replace=True); KS_eq=KSb.loc[take]
        elif len(KSb)>N_F: take=rng.choice(KSb.index, size=N_F, replace=False); KS_eq=KSb.loc[take]
        else: KS_eq=KSb

    feats=FSb.columns.tolist()
    KS_eq=KS_eq.reindex(columns=feats).fillna(0.0)
    kk=KS_eq.abs().mean(axis=0) if len(KS_eq)>0 else pd.Series(np.nan, index=feats)

    # Ranks & save CSV
    rF=ff.rank(ascending=False, method="min").astype(int)
    rK=kk.rank(ascending=False, method="min") if np.isfinite(kk.fillna(0)).any() else pd.Series(np.nan, index=feats)
    cmp=pd.DataFrame({
        "Feature": feats,
        f"Fourier_{cfg.SUFFIX}_NOAGE": ff.values,
        f"Kernel_{sfx_short}_NOAGE":  kk.values,
        "Rank_Fourier": rF.values,
        "Rank_Kernel":  rK.values,
        "DeltaRank(K-F)": (rK-rF).values,
        "DeltaSHAP(F-K)": (ff-kk).values
    }).sort_values(["Rank_Fourier","Feature"])
    safe_save_csv(cmp, os.path.join(age_dir, f"bin{b}_rank.csv"), index=False)

    # Common top-K feature list (by average importance magnitude)
    mean_both=(ff.fillna(0).abs()+kk.fillna(0).abs())/2.0
    feats_plot=mean_both.sort_values(ascending=False).index.tolist()[:cfg.TOPK_BARS]

    sizes.append({"bin":b, "N_test":len(idx_bin), "N_F":N_F, "N_K_raw":len(KSb), "N_K_eq":len(KS_eq)})
    costs.append({"bin": b,
                  "Kernel_time_sec": round(kernel_time, 6), "Kernel_peak_MB": round(kernel_peak_mb, 3),
                  "Fourier_time_sec": round(fourier_time, 6), "Fourier_peak_MB": round(fourier_peak_mb, 3)})

    perbin_data[b] = {
        "ff": ff, "kk": kk, "feats_plot": feats_plot,
        "N_F": N_F, "N_K_eq": len(KS_eq)
    }

safe_save_csv(pd.DataFrame(sizes), os.path.join(age_dir, "bins_sizes.csv"), index=False)
safe_save_csv(pd.DataFrame(costs), os.path.join(age_dir, "bins_costs.csv"), index=False)

100%|██████████| 87/87 [09:02<00:00,  6.23s/it]


In [27]:
# %%
# ---------------- Plotting Per-Bin Bars (separate cell) ----------------
# Generate both linear-scale and log-scale bar plots; save as .png and .pdf
for b in list(cfg.AGE_BINS):
    if perbin_data.get(b) is None:
        continue
    ff   = perbin_data[b]["ff"]
    kk   = perbin_data[b]["kk"]
    feats_plot = perbin_data[b]["feats_plot"]
    N_F  = perbin_data[b]["N_F"]
    N_K  = perbin_data[b]["N_K_eq"]

    x = np.arange(len(feats_plot)); width = 0.42
    # Display names for x tick labels
    xticklabels = [display_name(f) for f in feats_plot]

    # ---------- Linear scale ----------
    plt.figure(figsize=(11,6))
    plt.bar(x-width/2, kk.reindex(feats_plot).values, width, label=f"Kernel-{sfx_short}  N={N_K}")
    plt.bar(x+width/2, ff.reindex(feats_plot).values, width, label=f"Fourier-{cfg.SUFFIX} N={N_F}")
    plt.xticks(x, xticklabels, rotation=35, ha="right")
    plt.ylabel("Mean Absolute SHAP (logit)")
    plt.title(f"Age bin {b} MeanAbsSHAP (Linear Scale)")
    plt.tight_layout(); plt.legend()
    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_linear.pdf"), fmt="pdf")
    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_linear.png"), fmt="png")
    plt.close()

    # ---------- Log scale ----------

    plt.figure(figsize=(11,6))
    plt.bar(x - width/2, kk.reindex(feats_plot).values, width, label=f"Kernel-{sfx_short}  N={N_K}")
    plt.bar(x + width/2, ff.reindex(feats_plot).values, width, label=f"Fourier-{cfg.SUFFIX} N={N_F}")

    # Increase tick and label fonts
    plt.xticks(x, xticklabels, rotation=35, ha="right", fontsize=15)
    plt.yticks(fontsize=14)
    plt.ylabel("Mean Absolute SHAP (log scale, logit)", fontsize=16)
    plt.xlabel("Features", fontsize=16)

    # Log scale and title
    plt.yscale("log")
    plt.title(f"Age bin {b} MeanAbsSHAP (Log Scale)", fontsize=18, weight="bold")

    # Legend with larger font
    plt.legend(fontsize=14, loc="best")

    plt.tight_layout()
    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_log.pdf"), fmt="pdf")
    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_log.png"), fmt="png")
    plt.close()

#    plt.figure(figsize=(11,6))
#    plt.bar(x-width/2, kk.reindex(feats_plot).values, width, label=f"Kernel-{sfx_short}  N={N_K}")
#    plt.bar(x+width/2, ff.reindex(feats_plot).values, width, label=f"Fourier-{cfg.SUFFIX} N={N_F}")
#    plt.xticks(x, xticklabels, rotation=35, ha="right")
#    plt.ylabel("Mean Absolute SHAP (log scale, logit)")
#    plt.yscale("log")
#    plt.title(f"Age bin {b} MeanAbsSHAP (Log Scale)")
#    plt.tight_layout(); plt.legend()
#    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_log.pdf"), fmt="pdf")
#    safe_save_fig(os.path.join(age_dir, f"bin{b}_bars_log.png"), fmt="png")
#    plt.close()

In [28]:
# %%
# ---------------- Composite Images for Bins 0–3 and 4–7 (separate cell) ----------------
def compose_panel(bins, save_name):
    import matplotlib.image as mpimg
    nrows = len(bins)
    fig, axes = plt.subplots(nrows=nrows, ncols=2, figsize=(16, 4*nrows))
    if nrows == 1:
        axes = np.array([axes])  # ensure 2D

    for i, b in enumerate(bins):
        lin_path = os.path.join(age_dir, f"bin{b}_bars_linear.png")
        log_path = os.path.join(age_dir, f"bin{b}_bars_log.png")

        # In case a bin was empty
        if not os.path.exists(lin_path) or not os.path.exists(log_path):
            axes[i,0].axis("off")
            axes[i,1].axis("off")
            continue

        img_lin = mpimg.imread(lin_path)
        img_log = mpimg.imread(log_path)

        axes[i,0].imshow(img_lin)
        axes[i,0].axis("off")
        axes[i,0].set_title(f"Age bin {b} — MeanAbsSHAP (Linear)")

        axes[i,1].imshow(img_log)
        axes[i,1].axis("off")
        axes[i,1].set_title(f"Age bin {b} — MeanAbsSHAP (Log)")

    plt.tight_layout()
    safe_save_fig(os.path.join(age_dir, f"{save_name}.pdf"), fmt="pdf")
    safe_save_fig(os.path.join(age_dir, f"{save_name}.png"), fmt="png")
    plt.close()

# First image: bins 0–3
compose_panel([0,1,2,3], "AgeBins_0_3")
# Second image: bins 4–7
compose_panel([4,5,6,7], "AgeBins_4_7")

print(f"[{now()}] Done! Outputs in: {os.path.abspath(cfg.outdir)}")
print(f"[Info] Per-age (NO-AGE) folder: {os.path.abspath(age_dir)}")

[2025-10-28 11:09:31] Done! Outputs in: C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948
[Info] Per-age (NO-AGE) folder: C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\age_NOAGE_TBG


In [29]:
# %%
# ---------------- EXTRA CELL: SHAP on Probability Scale (no retrain) ----------------
# This cell converts Fourier SHAP (logit scale) to probability-scale using local Jacobian p*(1-p).
# Optionally, it can recompute KernelSHAP directly on probability (exact) if you set the flag below.

DO_RECOMPUTE_KERNEL_PROB = False  # Set to True to recompute Kernel SHAP on probability (heavier)

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Safety checks for required variables from previous cells
_required = ["cfg","FS_noage","p_te","X_te","age_dir","df","age_col","sfx_short","perbin_data","display_name"]
for _name in _required:
    if _name not in globals():
        raise RuntimeError(f"Variable '{_name}' not found. Please run prior cells before this one.")

# Helper: save figs
def _savefig(path_no_ext: str):
    safe_save_fig(path_no_ext + ".pdf", fmt="pdf")
    safe_save_fig(path_no_ext + ".png", fmt="png")

# 1) PROBABILITY-SCALE FOURIER SHAP (fast, no recompute)
#    SHAP_p ~= p*(1-p) * SHAP_logit (per-instance row-wise scaling)
p_vec = p_te  # probabilities for test instances from the trained model
jac = p_vec * (1.0 - p_vec)  # sigma'(z)
FS_prob = FS_noage.mul(jac, axis=0)  # DataFrame aligned by index

# Save the per-instance probability-scale Fourier SHAP (optional)
safe_save_csv(FS_prob, os.path.join(cfg.outdir, "fs_noage_probability.csv"), index=True)

# 2) (OPTIONAL) PROBABILITY-SCALE KERNEL SHAP - exact via recomputation with link="identity"
KS_prob_perbin = {}  # b -> DataFrame (may be empty if not recomputed)
if DO_RECOMPUTE_KERNEL_PROB:
    try:
        masker_prob = shap.maskers.Independent(X_tr)
        expl_prob = shap.Explainer(NNWithEncoder(enc, nn).predict_proba, masker_prob,
                                   algorithm="kernel", link="identity")
        sfx_prob = "IND"
    except Exception:
        expl_prob = shap.KernelExplainer(NNWithEncoder(enc, nn).predict_proba, X_tr, link="identity")
        sfx_prob = "TBG"

    age_map = df[age_col].to_dict()
    rng = np.random.default_rng(cfg.random_state)

    for b in list(cfg.AGE_BINS):
        if perbin_data.get(b) is None:
            KS_prob_perbin[b] = pd.DataFrame()
            continue
        idx_bin = [i for i in X_te.index if age_map.get(i, np.nan)==b]
        if len(idx_bin)==0:
            KS_prob_perbin[b] = pd.DataFrame()
            continue
        X_bin = X_te.loc[idx_bin]
        n_sub = min(cfg.KERNEL_BIN_SUB_N, len(X_bin))
        X_sub = X_bin.sample(n=n_sub, random_state=cfg.random_state)

        shap_vals_prob = expl_prob.shap_values(X_sub, nsamples=cfg.KERNEL_NSAMPLES)
        # shap returns a list for class outputs; take positive class (index 1)
        if isinstance(shap_vals_prob, list):
            arr = np.asarray(shap_vals_prob[1])
        else:
            arr = np.asarray(shap_vals_prob)
        if arr.shape == (n_sub, X_sub.shape[1]):
            KS_prob = pd.DataFrame(arr, index=X_sub.index, columns=list(X_sub.columns))
        elif arr.shape == (X_sub.shape[1], n_sub):
            KS_prob = pd.DataFrame(arr.T, index=X_sub.index, columns=list(X_sub.columns))
        else:
            raise ValueError(f"Unexpected Kernel prob-SHAP shape: {arr.shape}")
        # Drop age feature to keep NO-AGE convention
        KS_prob = KS_prob.drop(columns=[age_col], errors="ignore")
        KS_prob_perbin[b] = KS_prob

# 3) Build per-bin MEAN ABSOLUTE SHAP on probability scale and plot (linear + log)
#    - Fourier: from FS_prob (fast)
#    - Kernel: from KS_prob_perbin[b] if recomputed, else we skip or note unavailability
age_bins = list(cfg.AGE_BINS)
for b in age_bins:
    pdata = perbin_data.get(b)
    if pdata is None:
        continue

    # Fourier mean |SHAP| on prob scale
    idx_bin = [i for i in FS_prob.index if df.loc[i, age_col] == b]
    FSb_prob = FS_prob.loc[idx_bin] if len(idx_bin)>0 else FS_prob.iloc[0:0]
    ff_prob = FSb_prob.abs().mean(axis=0) if len(FSb_prob)>0 else pd.Series(np.nan, index=FS_prob.columns)

    # Kernel prob-scale mean |SHAP| (only if recomputed)
    if DO_RECOMPUTE_KERNEL_PROB and not KS_prob_perbin[b].empty:
        # Equalize N with Fourier (same protocol as before)
        KS_prob = KS_prob_perbin[b]
        N_F = len(FSb_prob)
        if N_F == 0:
            KS_eq_prob = KS_prob.iloc[0:0]
        else:
            if len(KS_prob) < N_F:
                take = np.random.default_rng(cfg.random_state).choice(KS_prob.index, size=N_F, replace=True)
                KS_eq_prob = KS_prob.loc[take]
            elif len(KS_prob) > N_F:
                take = np.random.default_rng(cfg.random_state).choice(KS_prob.index, size=N_F, replace=False)
                KS_eq_prob = KS_prob.loc[take]
            else:
                KS_eq_prob = KS_prob

        feats_common = ff_prob.index.tolist()
        kk_prob = KS_eq_prob.reindex(columns=feats_common).fillna(0.0).abs().mean(axis=0)
    else:
        kk_prob = None  # not available unless recomputed

    # Choose features to display: top-K by average magnitude of what's available
    if kk_prob is not None:
        mean_both = (ff_prob.fillna(0).abs() + kk_prob.fillna(0).abs()) / 2.0
    else:
        mean_both = ff_prob.fillna(0).abs()
    feats_plot = mean_both.sort_values(ascending=False).index.tolist()[:cfg.TOPK_BARS]
    xticklabels = [display_name(f) for f in feats_plot]
    x = np.arange(len(feats_plot)); width = 0.42

    # Save a CSV with the prob-scale means
    out_tbl = {
        "Feature": feats_plot,
        "Fourier_prob_MeanAbsSHAP": ff_prob.reindex(feats_plot).values
    }
    if kk_prob is not None:
        out_tbl[f"Kernel_prob_{sfx_short}_MeanAbsSHAP"] = kk_prob.reindex(feats_plot).values
    out_df = pd.DataFrame(out_tbl)
    safe_save_csv(out_df, os.path.join(age_dir, f"bin{b}_prob_meanabs.csv"), index=False)

    # ----- Linear scale plot (probability SHAP) -----
    plt.figure(figsize=(11,6))
    if kk_prob is not None:
        plt.bar(x - width/2, kk_prob.reindex(feats_plot).values, width, label=f"Kernel-{sfx_short} (prob)")
        plt.bar(x + width/2, ff_prob.reindex(feats_plot).values, width, label=f"Fourier-{cfg.SUFFIX} (prob)")
    else:
        plt.bar(x, ff_prob.reindex(feats_plot).values, width=0.8, label=f"Fourier-{cfg.SUFFIX} (prob)")
    plt.xticks(x, xticklabels, rotation=35, ha="right")
    plt.ylabel("Mean Absolute SHAP (probability scale)")
    plt.title(f"Age bin {b} Probability-Scale SHAP (Linear)")
    plt.tight_layout(); plt.legend()
    _savefig(os.path.join(age_dir, f"bin{b}_bars_prob_linear"))
    plt.close()

    # ----- Log scale plot (probability SHAP) -----
    plt.figure(figsize=(11,6))
    if kk_prob is not None:
        plt.bar(x - width/2, kk_prob.reindex(feats_plot).values, width, label=f"Kernel-{sfx_short} (prob)")
        plt.bar(x + width/2, ff_prob.reindex(feats_plot).values, width, label=f"Fourier-{cfg.SUFFIX} (prob)")
    else:
        plt.bar(x, ff_prob.reindex(feats_plot).values, width=0.8, label=f"Fourier-{cfg.SUFFIX} (prob)")
    plt.xticks(x, xticklabels, rotation=35, ha="right")
    plt.ylabel("Mean Absolute SHAP (log scale, probability)")
    plt.yscale("log")
    plt.title(f"Age bin {b} Probability-Scale SHAP (Log)")
    plt.tight_layout(); plt.legend()
    _savefig(os.path.join(age_dir, f"bin{b}_bars_prob_log"))
    plt.close()

print("[Info] Probability-scale SHAP plots saved alongside per-bin CSVs in:", os.path.abspath(age_dir))
print("[Info] To compute exact KernelSHAP on probability, set DO_RECOMPUTE_KERNEL_PROB=True in this cell.")


[Info] Probability-scale SHAP plots saved alongside per-bin CSVs in: C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\age_NOAGE_TBG
[Info] To compute exact KernelSHAP on probability, set DO_RECOMPUTE_KERNEL_PROB=True in this cell.


In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# -----------------------------
# Data from the provided table
# -----------------------------
bins = ["[2,15]", "[16,26]", "[27,36]", "[37,44]", "[45,52]", "[53,60]", "[61,71]", "[72,82]"]

T_kernel = np.array([579.0208, 611.2701, 475.1646, 516.9249, 520.6108, 557.1801, 623.4067, 542.0964])
PM_kernel = np.array([11510.6790, 11510.6530, 11510.6300, 11510.6340, 11510.6330, 11510.6340, 11510.6430, 11510.7810])

T_fourier = np.array([3.2280e-3, 1.6310e-3, 1.4140e-3, 1.6810e-3, 1.4630e-3, 1.4590e-3, 1.3890e-3, 1.7550e-3])
PM_fourier = np.array([4.3000e-2, 2.7000e-2, 2.2000e-2, 2.3000e-2, 2.4000e-2, 2.5000e-2, 2.9000e-2, 2.4000e-2])

# -----------------------------
# Transform: signed log10 (not absolute)
# -----------------------------
def safe_log10(x):
    x = np.asarray(x, dtype=float)
    if np.any(x <= 0):
        raise ValueError("All values must be positive to take log10.")
    return np.log10(x)

T_kernel_log10  = safe_log10(T_kernel)
T_fourier_log10 = safe_log10(T_fourier)
PM_kernel_log10  = safe_log10(PM_kernel)
PM_fourier_log10 = safe_log10(PM_fourier)

# Axis tick formatter to show powers of 10 (original units)
def pow10_formatter(x, pos):
    # x is already log10(value)
    k = int(np.round(x))
    return r"$10^{%d}$" % k

# Compute global y-limits and integer ticks for both subplots
def compute_limits_and_ticks(*arrays):
    vmin = min(np.min(a) for a in arrays)
    vmax = max(np.max(a) for a in arrays)
    # Expand a little for headroom
    vpad = 0.15 * (vmax - vmin if vmax > vmin else 1.0)
    vmin -= vpad
    vmax += vpad
    # Use integer ticks in log10 space
    tick_min = int(np.floor(min(np.min(a) for a in arrays)))
    tick_max = int(np.ceil (max(np.max(a) for a in arrays)))
    ticks = np.arange(tick_min, tick_max + 1, 1, dtype=int)
    return vmin, vmax, ticks

time_vmin, time_vmax, time_ticks = compute_limits_and_ticks(T_kernel_log10, T_fourier_log10)
mem_vmin,  mem_vmax,  mem_ticks  = compute_limits_and_ticks(PM_kernel_log10, PM_fourier_log10)

# -----------------------------
# Plotting configuration
# -----------------------------
plt.rcParams.update({
    "figure.figsize": (14, 5.5),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

x = np.arange(len(bins))
bar_width = 0.38
offset = bar_width / 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# -----------------------------
# Left subplot: Time (log10 scale, signed)
# -----------------------------
ax = axes[0]
ax.set_title("Runtime per Bin (bars show log10(seconds))")

ax.bar(x - offset, T_kernel_log10,  width=bar_width, label="T. Kernel (sec)")
ax.bar(x + offset, T_fourier_log10, width=bar_width, label="T. Fourier (sec)")

ax.set_xticks(x)
ax.set_xticklabels(bins, rotation=20)
ax.set_ylabel("Seconds (log10 scale)")
ax.set_ylim(time_vmin, time_vmax)
ax.set_yticks(time_ticks)
ax.yaxis.set_major_formatter(FuncFormatter(pow10_formatter))
ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.7)  # reference: 10^0 = 1 second
ax.grid(True, axis="y", linewidth=0.6, alpha=0.5)
ax.legend()

# -----------------------------
# Right subplot: Peak Memory (log10 scale, signed)
# -----------------------------
ax = axes[1]
ax.set_title("Peak Memory per Bin (bars show log10(MB))")

ax.bar(x - offset, PM_kernel_log10,  width=bar_width, label="P. M. Kernel (MB)")
ax.bar(x + offset, PM_fourier_log10, width=bar_width, label="P. M. Fourier (MB)")

ax.set_xticks(x)
ax.set_xticklabels(bins, rotation=20)
ax.set_ylabel("MB (log10 scale)")
ax.set_ylim(mem_vmin, mem_vmax)
ax.set_yticks(mem_ticks)
ax.yaxis.set_major_formatter(FuncFormatter(pow10_formatter))
ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.7)  # reference: 10^0 = 1 MB
ax.grid(True, axis="y", linewidth=0.6, alpha=0.5)
ax.legend()

#fig.suptitle("Kernel vs Fourier — Time and Peak Memory by Bin (Signed log10 Bars)", y=1.02, fontsize=15)
fig.tight_layout()

# -----------------------------
# Save outputs
# -----------------------------
out_png = "kernel_vs_fourier_bars_signed_log10.png"
out_pdf = "kernel_vs_fourier_bars_signed_log10.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.close()

print(f"Saved figure to:\n - {os.path.abspath(out_png)}\n - {os.path.abspath(out_pdf)}")


Saved figure to:
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\kernel_vs_fourier_bars_signed_log10.png
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\kernel_vs_fourier_bars_signed_log10.pdf


In [32]:
"""
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Try to enable optional PDF fallback (not required if you use PNGs)
try:
    from pdf2image import convert_from_path
    PDF2IMAGE_AVAILABLE = True
except Exception:
    PDF2IMAGE_AVAILABLE = False

 ------------------------------------------------
 Expected base names; the script will try .png first
 (e.g., bin0_bars_log.png), then .pdf as fallback
 ------------------------------------------------
base_names = [f"bin{i}_bars_log" for i in range(8)]

 Optional per-panel titles
titles = [f"Age bin {i}" for i in range(8)]

def load_img(path_png: Path, path_pdf: Path):
    """
    Load an image as a numpy array. Prefer PNG; if not found and a PDF exists,
    try to rasterize the first page via pdf2image (if available).
    """
    if path_png.exists():
        return plt.imread(str(path_png))
    if path_pdf.exists():
        if not PDF2IMAGE_AVAILABLE:
            raise RuntimeError(
                f"Found PDF but no PNG for '{path_png.stem}'. "
                "Please convert the PDF to PNG (recommended) or install pdf2image + Poppler.\n"
                f"Missing PNG: {path_png}\nExisting PDF: {path_pdf}"
            )
        images = convert_from_path(str(path_pdf), dpi=200, first_page=1, last_page=1)
        return np.array(images[0])
    raise FileNotFoundError(f"Neither PNG nor PDF found for '{path_png.stem}'. "
                            f"Looked for:\n - {path_png}\n - {path_pdf}")

def main():
    # Resolve and load images
    imgs = []
    missing = []
    for name in base_names:
        png = Path(f"{name}.png")
        pdf = Path(f"{name}.pdf")
        try:
            imgs.append(load_img(png, pdf))
        except Exception as e:
            missing.append(str(e))

    if missing:
        msg = "\n\n".join(missing)
        raise SystemExit(f"[Error] Some panels could not be loaded:\n{msg}")

    # ----------------------------------------
    # Create tiled figure: 4 rows × 2 columns
    # ----------------------------------------
    nrows, ncols = 4, 2
    figsize = (12, 16)  # Adjust to your manuscript proportions
    plt.rcParams.update({
        "figure.figsize": figsize,
        "font.size": 12,
        "axes.titlesize": 12,
    })
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)

    for idx, img in enumerate(imgs):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_axis_off()
        #if idx < len(titles):
        #    ax.set_title(titles[idx])

    fig.tight_layout()
    out_png = "shap_logit_grid.png"
    out_pdf = "shap_logit_grid.pdf"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved:\n - {os.path.abspath(out_png)}\n - {os.path.abspath(out_pdf)}")

if __name__ == "__main__":
    main()
"""

Saved:
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\shap_logit_grid.png
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\shap_logit_grid.pdf


In [24]:
# %% [markdown]
# ## EXTRA CELL — Ever married sanity checks (ablation-free, conditional permutation & contrasts)
# This cell corroborates whether **ever_married** behaves like a proxy/false positive without retraining:
# 1) Conditional permutation test within rich strata (age_bin × gender × work_type × residence_type × BMI- and glucose-quartiles).
#    - If AUC/AP barely change under MANY conditional shuffles of ever_married, its predictive role is likely spurious/contextual.
# 2) Contrast vs. permuting a known strong driver (e.g., smoking_status or hypertension).
# Outputs: prints + CSV summary saved in cfg.outdir.

import numpy as np, pandas as pd, os
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- Safety checks (requires prior cells) ----------
_required = ["cfg","df","X_te","y_te","enc","nn","features","age_col","now"]
for _name in _required:
    if _name not in globals():
        raise RuntimeError(f"[EverMarriedTest] Missing '{_name}'. Run prior cells first.")

# ---------- Helper: find a column by (case-insensitive) name with underscores ignored ----------
def _find_col_like(cands, cols):
    cols_norm = { "".join(ch for ch in c.lower() if ch.isalnum()): c for c in cols }
    for q in cands:
        key = "".join(ch for ch in q.lower() if ch.isalnum())
        if key in cols_norm:
            return cols_norm[key]
    return None

# Try to resolve key columns in your coded DataFrame (df) / features
em_col = _find_col_like(["ever_married","evermarried","marital_status"], features)
sex_col = _find_col_like(["gender","sex"], features)
wk_col  = _find_col_like(["work_type","worktype","employment"], features)
res_col = _find_col_like(["residence_type","residencetype","residence"], features)
bmi_col = _find_col_like(["bmi","bodymassindex"], features)
glu_col = _find_col_like(["avg_glucose_level","glucose","averageglucoselevel"], features)
smk_col = _find_col_like(["smoking_status","smoking","smoker"], features)
htn_col = _find_col_like(["hypertension","htn"], features)

if em_col is None:
    raise RuntimeError("[EverMarriedTest] Could not locate the 'ever_married' column among features.")

# ---------- Build strata on the TEST set indices ----------
idx_te = list(X_te.index)
df_te  = df.loc[idx_te, features].copy()   # coded (discrete) features as used for modeling

# Add robust BMI/glucose quartiles if available
if bmi_col is not None:
    df_te["bmi_q"] = pd.qcut(df_te[bmi_col].astype(float), q=4, labels=False, duplicates="drop")
else:
    df_te["bmi_q"] = 0
if glu_col is not None:
    df_te["gluc_q"] = pd.qcut(df_te[glu_col].astype(float), q=4, labels=False, duplicates="drop")
else:
    df_te["gluc_q"] = 0

# Compose strata columns (use only those that exist)
strata_cols = [c for c in [age_col, sex_col, wk_col, res_col, "bmi_q", "gluc_q"] if c is not None]

# ---------- Baseline metrics on the original test set ----------
prob = lambda X_: nn.predict_proba(enc.transform(X_))[:,1]
p0   = prob(df_te[features])
auc0 = roc_auc_score(y_te, p0)
ap0  = average_precision_score(y_te, p0)

# ---------- Conditional permutation utility ----------
def _conditional_permute_series_inplace(df_, col, strata_cols, rng):
    """
    Permute 'col' within each stratum defined by 'strata_cols' (keeps joint structure of confounders).
    Operates IN-PLACE on df_.
    """
    if not strata_cols:
        df_.loc[:, col] = rng.permutation(df_[col].values)  # unstratified fallback
        return
    # groupby on strata; shuffle within each group
    groups = df_.groupby(strata_cols, sort=False).indices
    for _, idx in groups.items():
        vals = df_.iloc[idx][col].values.copy()
        rng.shuffle(vals)
        df_.iloc[idx, df_.columns.get_loc(col)] = vals

# ---------- Experiment parameters ----------
B = 120  # number of conditional permutations (increase if you want tighter p-values)
rng = np.random.default_rng(cfg.random_state)

# Also pick a strong contrast feature to show expected drop (if present)
contrast_col = None
for c_try in [smk_col, htn_col, glu_col, bmi_col]:
    if c_try is not None:
        contrast_col = c_try
        break

records = []

# 1) CONDITIONAL PERM on ever_married
for b in range(B):
    df_perm = df_te.copy()
    _conditional_permute_series_inplace(df_perm, em_col, strata_cols, rng)
    p = prob(df_perm[features])
    records.append({
        "which": "ever_married",
        "iter": b,
        "AUC": roc_auc_score(y_te, p),
        "AP":  average_precision_score(y_te, p)
    })

# 2) CONDITIONAL PERM on a strong driver (contrast), if available
if contrast_col is not None:
    for b in range(B):
        df_perm = df_te.copy()
        _conditional_permute_series_inplace(df_perm, contrast_col, strata_cols, rng)
        p = prob(df_perm[features])
        records.append({
            "which": f"contrast::{contrast_col}",
            "iter": b,
            "AUC": roc_auc_score(y_te, p),
            "AP":  average_precision_score(y_te, p)
        })

res = pd.DataFrame(records)

# ---------- Summaries and empirical p-values ----------
def _summarize(dfm, auc0, ap0):
    dAUC = dfm["AUC"] - auc0
    dAP  = dfm["AP"]  - ap0
    # Two-sided empirical p-values for "no effect"
    p_auc = (np.abs(dAUC) >= np.abs(0)).mean()  # (kept for completeness)
    # more informative: probability of |drop| ≥ observed drop when permuting
    # but observed drop against itself is zero; instead report mean & 95% CI
    return {
        "AUC_base": auc0,
        "AP_base":  ap0,
        "AUC_perm_mean": dfm["AUC"].mean(),
        "AUC_perm_std":  dfm["AUC"].std(ddof=1),
        "AP_perm_mean":  dfm["AP"].mean(),
        "AP_perm_std":   dfm["AP"].std(ddof=1),
        "dAUC_mean":     dAUC.mean(),
        "dAUC_ci95":     (np.percentile(dAUC, 2.5), np.percentile(dAUC, 97.5)),
        "dAP_mean":      dAP.mean(),
        "dAP_ci95":      (np.percentile(dAP, 2.5),  np.percentile(dAP, 97.5)),
    }

out_rows = []
em_df = res[res["which"]=="ever_married"]
out_em = _summarize(em_df, auc0, ap0); out_em["which"] = "ever_married"; out_rows.append(out_em)

if contrast_col is not None:
    ct_df = res[res["which"]==f"contrast::{contrast_col}"]
    out_ct = _summarize(ct_df, auc0, ap0); out_ct["which"] = f"contrast::{contrast_col}"; out_rows.append(out_ct)

summary = pd.DataFrame(out_rows)[["which","AUC_base","AUC_perm_mean","AUC_perm_std","dAUC_mean","dAUC_ci95",
                                  "AP_base","AP_perm_mean","AP_perm_std","dAP_mean","dAP_ci95"]]

# ---------- Save & print ----------
csv_path = os.path.join(cfg.outdir, "ever_married_conditional_permutation_summary.csv")
summary.to_csv(csv_path, index=False)

print("\n[EverMarriedTest] Conditional permutation within strata:", strata_cols or "(no strata)")
print(f"Baseline: AUC={auc0:.4f} | AP={ap0:.4f}")
print(summary.to_string(index=False))
print(f"[EverMarriedTest] Saved summary to: {os.path.abspath(csv_path)}")

# ---------- Quick heuristic interpretation ----------
msg = []
dAUC_em = summary.loc[summary["which"]=="ever_married","dAUC_mean"].values[0]
dAP_em  = summary.loc[summary["which"]=="ever_married","dAP_mean"].values[0]
msg.append(f"[Heuristic] Conditional permuting 'ever_married' changes AUC on average by {dAUC_em:.4f} "
           f"and AP by {dAP_em:.4f}. Small (|ΔAUC|<0.005 and |ΔAP|<0.002) suggests weak unique signal/proxy.")
if contrast_col is not None:
    dAUC_ct = summary.loc[summary["which"]==f"contrast::{contrast_col}","dAUC_mean"].values[0]
    dAP_ct  = summary.loc[summary["which"]==f"contrast::{contrast_col}","dAP_mean"].values[0]
    msg.append(f"[Heuristic] For contrast feature '{contrast_col}', mean ΔAUC={dAUC_ct:.4f}, ΔAP={dAP_ct:.4f} "
               f"(larger drops than 'ever_married' reinforce the proxy hypothesis).")
print("\n".join(msg))



[EverMarriedTest] Conditional permutation within strata: ['age', 'gender', 'work_type', 'Residence_type', 'bmi_q', 'gluc_q']
Baseline: AUC=0.8283 | AP=0.2027
                   which  AUC_base  AUC_perm_mean  AUC_perm_std  dAUC_mean                                     dAUC_ci95  AP_base  AP_perm_mean  AP_perm_std  dAP_mean                                      dAP_ci95
            ever_married  0.828269       0.823099      0.004590  -0.005170 (-0.012505267100519649, 0.003325296128095944) 0.202702      0.191033     0.013402 -0.011668  (-0.03368906419595257, 0.010917723733741736)
contrast::smoking_status  0.828269       0.820791      0.004488  -0.007478 (-0.017473898590758, -0.00037513460367987416) 0.202702      0.202317     0.019199 -0.000385 (-0.030889007626950384, 0.046685563595646296)
[EverMarriedTest] Saved summary to: C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\ever_married_conditional_p

In [25]:
# %% [markdown]
# ## EXTRA CELL — Per-bin conditional permutation test for *Ever married* (and contrast feature)
# This cell repeats the conditional permutation analysis **within each age bin**.
# Goal: Verify whether *Ever married* has unique predictive value per bin (esp. Bin 0: [2,15]).
# - For each bin b, we:
#   (i) restrict to the test instances with age_bin == b,
#   (ii) build rich strata (gender × work_type × residence_type × BMI- and glucose-quartiles),
#   (iii) conditionally permute *ever_married* within strata (B times) and recompute AUC/AP,
#   (iv) optionally do the same for a strong contrast feature (e.g., smoking_status or hypertension),
#   (v) summarize mean ΔAUC, ΔAP and 95% empirical CIs.
# Results are saved to CSV in `age_dir` and a compact summary is printed (with special note for Bin 0).

import numpy as np, pandas as pd, os
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- Safety checks: relies on variables defined in your previous cells ----------
_required = ["cfg","df","X_te","y_te","enc","nn","features","age_col","age_dir","now"]
for _name in _required:
    if _name not in globals():
        raise RuntimeError(f"[PerBinEverMarriedTest] Missing '{_name}'. Run prior cells first.")

# ---------- Small utilities ----------
def _find_col_like(cands, cols):
    """Find a column among `cols` that matches any name in `cands` ignoring case and underscores."""
    cols_norm = {"".join(ch for ch in c.lower() if ch.isalnum()): c for c in cols}
    for q in cands:
        key = "".join(ch for ch in q.lower() if ch.isalnum())
        if key in cols_norm:
            return cols_norm[key]
    return None

def _conditional_permute_series_inplace(df_, col, strata_cols, rng):
    """Permute `col` within each group defined by `strata_cols` (in place)."""
    if not strata_cols:
        df_.loc[:, col] = rng.permutation(df_[col].values)
        return
    groups = df_.groupby(strata_cols, sort=False).indices
    for _, idx in groups.items():
        vals = df_.iloc[idx][col].values.copy()
        rng.shuffle(vals)
        df_.iloc[idx, df_.columns.get_loc(col)] = vals

# Resolve key columns present in your coded `features`
em_col = _find_col_like(["ever_married","evermarried","marital_status"], features)
sex_col = _find_col_like(["gender","sex"], features)
wk_col  = _find_col_like(["work_type","worktype","employment"], features)
res_col = _find_col_like(["residence_type","residencetype","residence"], features)
bmi_col = _find_col_like(["bmi","bodymassindex"], features)
glu_col = _find_col_like(["avg_glucose_level","glucose","averageglucoselevel"], features)
smk_col = _find_col_like(["smoking_status","smoking","smoker"], features)
htn_col = _find_col_like(["hypertension","htn"], features)

if em_col is None:
    raise RuntimeError("[PerBinEverMarriedTest] Could not locate the 'ever_married' feature in `features`.")

# Probability function using the trained NN + encoder (already in memory)
prob = lambda X_: nn.predict_proba(enc.transform(X_))[:,1]

# Prepare per-bin testing DataFrames (restrict to test indices only)
idx_te = list(X_te.index)
df_te  = df.loc[idx_te, features].copy()
y_te_s = pd.Series(y_te, index=idx_te, name="y")

# Experiment settings
B = 120  # number of conditional permutations per (bin, feature)
rng = np.random.default_rng(cfg.random_state)

# Choose a contrast feature expected to cause a noticeable drop when permuted
contrast_col = None
for c_try in [smk_col, htn_col, glu_col, bmi_col]:
    if c_try is not None:
        contrast_col = c_try
        break

records = []  # will collect all per-iteration results
summ_rows = []  # compact per-bin summaries

bins_to_run = list(cfg.AGE_BINS)
print(f"[{now()}] Per-bin conditional permutation test starting... B={B} permutations per feature.")

for b in bins_to_run:
    # ---- Select test instances for this bin ----
    idx_bin = [i for i in idx_te if df.loc[i, age_col] == b]
    if len(idx_bin) == 0:
        print(f"[PerBin] Bin {b}: no test instances. Skipping.")
        continue

    dfb = df_te.loc[idx_bin].copy()
    yb  = y_te_s.loc[idx_bin].to_numpy()

    # Build quartiles (computed within the bin)
    if bmi_col is not None:
        dfb["bmi_q"] = pd.qcut(dfb[bmi_col].astype(float), q=4, labels=False, duplicates="drop")
    else:
        dfb["bmi_q"] = 0
    if glu_col is not None:
        dfb["gluc_q"] = pd.qcut(dfb[glu_col].astype(float), q=4, labels=False, duplicates="drop")
    else:
        dfb["gluc_q"] = 0

    # Strata *within* bin (do NOT include age, since all are same bin now)
    strata_cols = [c for c in [sex_col, wk_col, res_col, "bmi_q", "gluc_q"] if c is not None]

    # Baseline metrics for this bin
    p0 = prob(dfb[features])
    auc0 = roc_auc_score(yb, p0) if len(np.unique(yb)) > 1 else np.nan
    ap0  = average_precision_score(yb, p0) if len(np.unique(yb)) > 1 else np.nan

    # Warn if bin is very small or nearly separable
    if len(dfb) < 40:
        print(f"[PerBin] Bin {b}: small sample (N={len(dfb)}). CIs may be wide.")
    if np.isnan(auc0) or np.isnan(ap0):
        print(f"[PerBin] Bin {b}: baseline metrics undefined (single-class?). Skipping.")
        continue

    # 1) Permute *ever_married* within strata
    for it in range(B):
        dfp = dfb.copy()
        _conditional_permute_series_inplace(dfp, em_col, strata_cols, rng)
        p = prob(dfp[features])
        records.append({
            "bin": b, "which": "ever_married", "iter": it,
            "AUC": roc_auc_score(yb, p), "AP": average_precision_score(yb, p),
            "AUC_base": auc0, "AP_base": ap0, "N_bin": len(dfb)
        })

    # 2) Permute a contrast feature (if available)
    if contrast_col is not None:
        for it in range(B):
            dfp = dfb.copy()
            _conditional_permute_series_inplace(dfp, contrast_col, strata_cols, rng)
            p = prob(dfp[features])
            records.append({
                "bin": b, "which": f"contrast::{contrast_col}", "iter": it,
                "AUC": roc_auc_score(yb, p), "AP": average_precision_score(yb, p),
                "AUC_base": auc0, "AP_base": ap0, "N_bin": len(dfb)
            })

    # Build a compact summary row per bin for quick reading
    rec_bin = pd.DataFrame([r for r in records if r["bin"] == b])
    for w in ["ever_married"] + ([f"contrast::{contrast_col}"] if contrast_col is not None else []):
        sub = rec_bin[rec_bin["which"] == w]
        if len(sub) == 0:
            continue
        dAUC = sub["AUC"].to_numpy() - sub["AUC_base"].to_numpy()
        dAP  = sub["AP"].to_numpy()  - sub["AP_base"].to_numpy()
        summ_rows.append({
            "bin": b, "which": w, "N_bin": int(sub["N_bin"].iloc[0]),
            "AUC_base": float(sub["AUC_base"].iloc[0]),
            "AUC_perm_mean": float(sub["AUC"].mean()), "AUC_perm_std": float(sub["AUC"].std(ddof=1)),
            "dAUC_mean": float(dAUC.mean()),
            "dAUC_ci95_low": float(np.percentile(dAUC, 2.5)),
            "dAUC_ci95_high": float(np.percentile(dAUC, 97.5)),
            "AP_base": float(sub["AP_base"].iloc[0]),
            "AP_perm_mean": float(sub["AP"].mean()), "AP_perm_std": float(sub["AP"].std(ddof=1)),
            "dAP_mean": float(dAP.mean()),
            "dAP_ci95_low": float(np.percentile(dAP, 2.5)),
            "dAP_ci95_high": float(np.percentile(dAP, 97.5)),
        })

# ---------- Save outputs ----------
res_df   = pd.DataFrame(records)
summ_df  = pd.DataFrame(summ_rows).sort_values(["bin","which"])
out_res  = os.path.join(age_dir, "perbin_conditional_permutation_records.csv")
out_summ = os.path.join(age_dir, "perbin_conditional_permutation_summary.csv")
res_df.to_csv(out_res, index=False)
summ_df.to_csv(out_summ, index=False)

# ---------- Print compact summary and flag Bin 0 ----------
print(f"\n[PerBinEverMarriedTest] Saved records to:\n - {os.path.abspath(out_res)}")
print(f"[PerBinEverMarriedTest] Saved summary to:\n - {os.path.abspath(out_summ)}\n")

# Pretty print per-bin results for ever_married
if not summ_df.empty:
    em_only = summ_df[summ_df["which"]=="ever_married"].copy()
    cols = ["bin","N_bin","AUC_base","AUC_perm_mean","dAUC_mean","dAUC_ci95_low","dAUC_ci95_high",
            "AP_base","AP_perm_mean","dAP_mean","dAP_ci95_low","dAP_ci95_high"]
    print("[PerBinEverMarriedTest] Ever married — per-bin summary:")
    print(em_only[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Special note for Bin 0 (age ~ [2,15])
    if (em_only["bin"]==0).any():
        row0 = em_only[em_only["bin"]==0].iloc[0]
        msg = (f"\n[Note Bin 0] age≈[2,15]. Ever married as top feature is implausible. "
               f"Here, conditional permutation yields ΔAUC={row0['dAUC_mean']:.4f} "
               f"(95% CI [{row0['dAUC_ci95_low']:.4f}, {row0['dAUC_ci95_high']:.4f}]) and "
               f"ΔAP={row0['dAP_mean']:.4f} (95% CI [{row0['dAP_ci95_low']:.4f}, {row0['dAP_ci95_high']:.4f}]). "
               f"Small/near-zero drops would indicate proxy/artefact (e.g., coding leakage or correlation with other strata).")
        print(msg)

print(f"\n[{now()}] Per-bin conditional permutation test completed.")


[2025-10-28 09:31:50] Per-bin conditional permutation test starting... B=120 permutations per feature.
[PerBin] Bin 0: baseline metrics undefined (single-class?). Skipping.
[PerBin] Bin 1: baseline metrics undefined (single-class?). Skipping.
[PerBin] Bin 2: baseline metrics undefined (single-class?). Skipping.

[PerBinEverMarriedTest] Saved records to:
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\age_NOAGE_TBG\perbin_conditional_permutation_records.csv
[PerBinEverMarriedTest] Saved summary to:
 - C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\age_NOAGE_TBG\perbin_conditional_permutation_summary.csv

[PerBinEverMarriedTest] Ever married — per-bin summary:
 bin  N_bin  AUC_base  AUC_perm_mean  dAUC_mean  dAUC_ci95_low  dAUC_ci95_high  AP_base  AP_perm_mean  dAP_mean  dAP_ci95_low  dAP_ci95_high
   3

In [26]:
# %% [markdown]
# ## EXTRA CELL — Per-bin label counts & "Ever married" prevalence sanity checks
# This cell summarizes, for the TEST set only:
#   • Per-bin counts of positives/negatives
#   • Prevalence of `ever_married` per bin
#   • Flags bins that are single-class (AUC/AP undefined risk)
#   • (If a continuous age column exists) lists rows where age < 16 but ever_married == 1
# Outputs are printed and also saved as CSV in `age_dir`.

import numpy as np, pandas as pd, os

# ---------- Safety checks ----------
_required = ["cfg","df","X_te","y_te","features","age_col","age_dir","now"]
for _name in _required:
    if _name not in globals():
        raise RuntimeError(f"[PerBinSanity] Missing '{_name}'. Run the previous cells first.")

# ---------- Helpers ----------
def _find_col_like(cands, cols):
    """Return the first column in `cols` that matches one of `cands` ignoring case/underscores."""
    norm = {"".join(ch for ch in c.lower() if ch.isalnum()): c for c in cols}
    for q in cands:
        key = "".join(ch for ch in q.lower() if ch.isalnum())
        if key in norm:
            return norm[key]
    return None

def _find_continuous_age(df_all):
    """Try to find a continuous age column (many unique values), different from the bin index."""
    cands = [c for c in df_all.columns if "age" in c.lower()]
    # Prefer numeric with > 20 unique values (likely continuous), not equal to age_col
    for c in cands:
        if c == age_col: 
            continue
        s = df_all[c]
        if np.issubdtype(s.dtype, np.number) and s.nunique(dropna=True) > 20:
            return c
    # Fallback: None (we'll only use age bins)
    return None

# Resolve columns
em_col  = _find_col_like(["ever_married","evermarried","marital_status"], features)
sex_col = _find_col_like(["gender","sex"], features)

if em_col is None:
    raise RuntimeError("[PerBinSanity] Could not locate 'ever_married' among `features`.")

# Build TEST-frame with features + target
idx_te = list(X_te.index)
df_te  = df.loc[idx_te, features].copy()
y_te_s = pd.Series(y_te, index=idx_te, name="y")

# Try to find a continuous age column for additional plausibility checks
age_cont_col = _find_continuous_age(df)

# ---------- Per-bin summary ----------
rows = []
for b in list(cfg.AGE_BINS):
    idx_bin = [i for i in idx_te if df.loc[i, age_col] == b]
    N = len(idx_bin)
    if N == 0:
        rows.append({
            "bin": b, "N_test": 0, "n_pos": np.nan, "n_neg": np.nan,
            "pos_rate": np.nan, "em_ones": np.nan, "em_rate": np.nan,
            "single_class": None
        })
        continue

    yb = y_te_s.loc[idx_bin].to_numpy()
    n_pos = int(np.sum(yb))
    n_neg = int(N - n_pos)
    single_class = (n_pos == 0 or n_neg == 0)

    em_vals = df_te.loc[idx_bin, em_col]
    # If em is categorical/one-hot-coded numeric ints, treat nonzero as 1
    if not np.issubdtype(em_vals.dtype, np.number):
        em_bin = (em_vals.astype(str).str.strip().str.lower().isin(["1","yes","true","married","ever"])).astype(int)
    else:
        em_bin = (em_vals.to_numpy() != 0).astype(int)

    em_ones = int(em_bin.sum())
    em_rate = float(em_bin.mean())
    pos_rate = float(n_pos / max(1, N))

    rows.append({
        "bin": b, "N_test": N, "n_pos": n_pos, "n_neg": n_neg,
        "pos_rate": round(pos_rate, 6),
        "em_ones": em_ones, "em_rate": round(em_rate, 6),
        "single_class": single_class
    })

summary = pd.DataFrame(rows).sort_values("bin")
out_csv = os.path.join(age_dir, "perbin_labelcounts_evermarried_prevalence.csv")
summary.to_csv(out_csv, index=False)

print(f"[{now()}] Per-bin TEST-set summary (label counts & 'ever married' prevalence):")
print(summary.to_string(index=False))
print(f"[PerBinSanity] Saved CSV to: {os.path.abspath(out_csv)}")

# ---------- Warnings & sanity notes ----------
notes = []
# 1) Single-class bins (AUC/AP not computable)
sc_bins = summary.loc[summary["single_class"]==True, "bin"].tolist()
if sc_bins:
    notes.append(f"[Warn] Single-class TEST bins detected (AUC/AP undefined): {sc_bins}. "
                 f"Consider stratified splitting by (age_bin × label) or cross-validation with per-bin scoring.")

# 2) Implausible 'ever married' in youngest bin (bin 0 ~ ages [2,15])
row0 = summary[summary["bin"]==0]
if not row0.empty and row0.iloc[0]["N_test"] > 0:
    em_rate0 = row0.iloc[0]["em_rate"]
    if em_rate0 > 0:
        notes.append(f"[Warn] Bin 0 (≈[2,15]) shows ever_married prevalence of {em_rate0:.3f}. "
                     f"This is implausible; check coding/one-hot mappings and raw data.")

# 3) If continuous age is available, list rows age<16 with ever_married==1
if age_cont_col is not None:
    df_age = df.loc[idx_te, [age_cont_col] + [em_col]].copy()
    # Normalize ever_married to binary 0/1 as above
    if not np.issubdtype(df_age[em_col].dtype, np.number):
        em_bin_all = (df_age[em_col].astype(str).str.strip().str.lower().isin(["1","yes","true","married","ever"])).astype(int)
    else:
        em_bin_all = (df_age[em_col].to_numpy() != 0).astype(int)
    bad_idx = df_age.index[(df_age[age_cont_col] < 16) & (em_bin_all == 1)].tolist()
    if bad_idx:
        bad_tbl = df.loc[bad_idx, [age_cont_col, em_col] + ([sex_col] if sex_col else [])].copy()
        bad_csv = os.path.join(age_dir, "implausible_evermarried_age_lt16.csv")
        bad_tbl.to_csv(bad_csv, index=True)
        notes.append(f"[Warn] Found {len(bad_idx)} TEST rows with {age_cont_col} < 16 and ever_married==1. "
                     f"Exported details to: {os.path.abspath(bad_csv)}")
    else:
        notes.append(f"[Info] No TEST rows with {age_cont_col} < 16 and ever_married==1.")

# 4) General tip if 'ever_married' is frequently nonzero in young bins
if any((summary["bin"]<=1) & (summary["em_rate"] > 0)):
    notes.append("[Tip] If encoding used one-hot columns for marital categories, ensure the column you use for "
                 "'ever_married' is not a residual dummy (e.g., 'Unknown' or 'Never married' recoded to 1).")

if notes:
    print("\n".join(notes))
else:
    print("[PerBinSanity] No obvious issues detected.")



[2025-10-28 09:45:06] Per-bin TEST-set summary (label counts & 'ever married' prevalence):
 bin  N_test  n_pos  n_neg  pos_rate  em_ones  em_rate  single_class
   0      93      0     93  0.000000        0 0.000000          True
   1      99      0     99  0.000000       10 0.101010          True
   2      77      0     77  0.000000       47 0.610390          True
   3      83      1     82  0.012048       69 0.831325         False
   4      84      2     82  0.023810       75 0.892857         False
   5      90      5     85  0.055556       84 0.933333         False
   6     107      8     99  0.074766      104 0.971963         False
   7      87     15     72  0.172414       75 0.862069         False
[PerBinSanity] Saved CSV to: C:\Users\roberto.morales\Dropbox\Research\Research 2025\Machine Learning\Codigos\SHAP Fourier\Experiment 3 Healthcare 02\OutExp2_1761289948\age_NOAGE_TBG\perbin_labelcounts_evermarried_prevalence.csv
[Warn] Single-class TEST bins detected (AUC/AP undefined): 